# 02 · Indexer backend selector — `python` vs `c-omp`

`midas-pipeline` ships two interchangeable indexer backends behind one
flag, `--indexer-backend {python, c-omp}`:

- **`python`** — the in-process PyTorch/Numba indexer
  (`midas_index`). CPU/CUDA/MPS, autograd-friendly, no compiler needed.
- **`c-omp`** — the bundled OpenMP C binary (`midas_indexer`), built by
  scikit-build-core at pip-install time when a compiler is available.

Both read the **same** unified on-disk contract written by the binning
stage: a 10-column `Spots.bin` (col 9 = ScanNr, 0 for FF) plus an
`int64`-pair `Data.bin` / `nData.bin`. This notebook runs the FF
pipeline once, then drives both backends over the identical binned
inputs and shows they read the same data and agree.

> **Runtime** ~1–1.5 min on CPU.

In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np

from midas_pipeline import __version__
from midas_index import backend_c
print('midas-pipeline', __version__)
print('c-omp backend available:', backend_c.available())
if backend_c.available():
    print('c-omp binary:', backend_c.binary_path())

midas-pipeline 0.2.1
c-omp backend available: True
c-omp binary: /Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/midas_index/bin/midas_indexer


## 1. Forward-simulate + run upstream stages (shared inputs)

We generate one synthetic FF dataset and run the orchestrator through
**binning** (skipping indexing and after). The binned `Spots.bin` /
`Data.bin` / `nData.bin` become the shared inputs that both backends
will index. Running upstream once keeps the comparison apples-to-apples.

In [2]:
from midas_ff_pipeline.testing import generate_synthetic_dataset

MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or Path.home() / 'opt' / 'MIDAS')
PARAMS_TEMPLATE = MIDAS_HOME / 'FF_HEDM' / 'Example' / 'Parameters.txt'

WORK = Path(tempfile.mkdtemp(prefix='midas_pipeline_idx_'))
zarr = generate_synthetic_dataset(
    out_dir=WORK / 'sim', params_template=PARAMS_TEMPLATE,
    n_grains=20, n_cpus=4,
)

UPSTREAM = WORK / 'upstream'
cmd = [
    sys.executable, '-m', 'midas_pipeline', 'run', '--scan-mode', 'ff',
    '--params', str(WORK / 'sim' / PARAMS_TEMPLATE.name),
    '--result', str(UPSTREAM), '--zarr', str(zarr),
    '--n-cpus', '4', '--device', 'cpu', '--dtype', 'float64',
    '--skip', 'indexing', '--skip', 'refinement',
    '--skip', 'process_grains', '--skip', 'consolidation',
]
t0 = time.time()
proc = subprocess.run(cmd, capture_output=True, text=True)
print(f'upstream exit={proc.returncode} ({time.time() - t0:.1f}s)')
assert proc.returncode == 0, proc.stderr[-2000:]

layer = UPSTREAM / 'LayerNr_1'
for n in ('Spots.bin', 'Data.bin', 'nData.bin'):
    print(f'  {n:12s}', f'{(layer / n).stat().st_size:>14,} bytes')
assert (layer / 'Data.bin').stat().st_size > 0

upstream exit=0 (36.5s)
  Spots.bin           170,880 bytes
  Data.bin          3,258,848 bytes
  nData.bin     1,036,800,000 bytes


## 2. Run the indexer directly with each backend

We invoke `midas_index` over the binned `LayerNr_1/` with each backend.
The CLI argv is the IndexerOMP-compatible
`<paramstest> <block> <n_blocks> <n_seeds> <n_cpus>`. We point the C
binary's `OutputFolder` at a per-backend directory so the consolidated
output lands somewhere we can inspect.

In [3]:
def run_index(backend, tag):
    out = WORK / f'idx_{tag}'
    out.mkdir(parents=True, exist_ok=True)
    n_seeds = sum(1 for _ in (layer / 'SpotsToIndex.csv').open())
    if backend == 'python':
        cmd = [sys.executable, '-m', 'midas_index', str(layer / 'paramstest.txt'),
               '0', '1', str(n_seeds), '4', '--device', 'cpu', '--dtype', 'float64',
               '--group-size', '4']
    else:
        cmd = [str(backend_c.binary_path()), str(layer / 'paramstest.txt'),
               '0', '1', str(n_seeds), '4']
    t0 = time.time()
    p = subprocess.run(cmd, cwd=str(layer), capture_output=True, text=True)
    dt = time.time() - t0
    return p, dt

py_proc, py_dt = run_index('python', 'py')
print(f'[python] exit={py_proc.returncode} ({py_dt:.2f}s)')
print('  ', py_proc.stderr.strip().splitlines()[-1] if py_proc.stderr.strip() else '')
assert py_proc.returncode == 0, py_proc.stderr[-1500:]

[python] exit=0 (7.12s)
   midas-index 0.5.2: block 0/1 completed. 93 seeds indexed -> ./IndexBest.bin


In [4]:
if backend_c.available():
    c_proc, c_dt = run_index('c-omp', 'c')
    print(f'[c-omp] exit={c_proc.returncode} ({c_dt:.2f}s)')
    # The C binary echoes what it read from the binned files to stdout.
    for line in c_proc.stdout.splitlines():
        if any(k in line for k in ('nSpots', 'DataSize', 'nDataSize',
                                   'Binning', 'Mode', 'Finished')):
            print('  ', line.strip())
    assert c_proc.returncode == 0, c_proc.stderr[-1500:]
else:
    print('c-omp binary not built in this environment — skipping the C run.')
    print('Re-install midas-index with a C/OpenMP toolchain to enable it.')
    c_proc = None

[c-omp] exit=0 (0.61s)
   nSpots = 2136
   DataSize: 3258848 8 Nelems: 407356
   nDataSize: 1036800000 8 Nelems: 129600000
   Binning: ring=5 eta=3600 ome=3600 total=64800000
   Mode: FF | numScans=1 nVoxels=109 blockNr=0 nBlocks=1 [startVoxel,endVoxel)=[0,109) numProcs=4
   Finished. Mode=FF nVoxels=109 block=[0,109). Time: 0.555202s.


## 3. Do the backends agree?

The two backends share one byte-level contract, so the decisive check is
that the C binary reads the **same** observed-spot count and per-bin
table that the Python backend indexes. The Python backend additionally
materialises `IndexBest.bin`; we read back its solution count.

(The C binary writes consolidated outputs under its own `OutputFolder`
convention rather than `IndexBest.bin`; both completing cleanly over the
identical `Spots.bin` / `Data.bin` is the agreement we assert here.)

In [5]:
# Python backend: number of seeds with a solution.
ib = np.fromfile(layer / 'IndexBest.bin', dtype=np.float64).reshape(-1, 15)
py_solved = int((ib[:, 14] > 0).sum())
print(f'python backend: {py_solved} / {ib.shape[0]} seeds solved')

# What the indexer read from disk (10-col Spots.bin, int64 Data.bin).
spots = np.fromfile(layer / 'Spots.bin', dtype=np.float64)
n_spots = spots.size // 10
data = np.fromfile(layer / 'Data.bin', dtype=np.int64).reshape(-1, 2)
print(f'shared inputs : Spots.bin -> {n_spots} spots (10-col), '
      f'Data.bin -> {data.shape[0]} (spot, scan) entries')

if c_proc is not None:
    # The C binary prints 'nSpots = <N>' — confirm it matches.
    c_nspots = None
    for line in c_proc.stdout.splitlines():
        if line.strip().startswith('nSpots'):
            c_nspots = int(line.split('=')[1])
    print(f'c-omp read    : nSpots = {c_nspots}')
    assert c_nspots == n_spots, (c_nspots, n_spots)
    print('AGREE: both backends read the same', n_spots, 'observed spots from the unified Spots.bin.')
else:
    print('c-omp unavailable — python backend solved', py_solved, 'seeds.')

python backend: 93 / 109 seeds solved
shared inputs : Spots.bin -> 2136 spots (10-col), Data.bin -> 203678 (spot, scan) entries
c-omp read    : nSpots = 2136
AGREE: both backends read the same 2136 observed spots from the unified Spots.bin.


In [6]:
shutil.rmtree(WORK, ignore_errors=True)
print('cleaned', WORK)

cleaned /var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/midas_pipeline_idx_bxe97hf0
